In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

FIG = Path("../figures/annotation")
RES = Path("../results/cell_annotation")
FIG.mkdir(parents=True, exist_ok=True)
(RES / "v2").mkdir(parents=True, exist_ok=True)
Path("../figures/qc").mkdir(parents=True, exist_ok=True)
Path("../results/qc").mkdir(parents=True, exist_ok=True)
Path("../figure_data").mkdir(parents=True, exist_ok=True)


In [2]:
adata = sc.read_h5ad("../data_interim/dimensionality_reduced.h5ad", backed="r")

In [3]:
PANELS = {
    "t_cell": ["Cd3d", "Cd3e", "Cd3g", "Lck", "Lat", "Cd247", "Themis"],
    "b_cell": ["Cd79a", "Cd79b", "Cd19", "Ms4a1", "Pax5", "Cd22", "Bank1", "Cd37", "Cd40", "Spib"],
    "nk_cell": ["Ncr1", "Klrd1", "Klrb1c", "Klrk1", "Eomes", "Il2rb", "Nkg7", "Prf1", "Gzmb", "Ccl5"],
    "monocyte": ["Ly6c2", "Ccr2", "Vcan", "Plac8", "Lyz2", "Itgam", "Csf1r", "Sell", "S100a8", "S100a9"],
    "dendritic": ["Flt3", "Zbtb46", "Itgax", "H2-Ab1", "H2-Aa", "Cd74", "Clec10a", "Xcr1", "Clec9a", "Sirpa"],
    "endothelial": ["Pecam1", "Cdh5", "Kdr", "Flt1", "Vwf", "Emcn", "Cldn5", "Esam", "Egfl7", "Tek", "Robo4", "Klf2"],
    "macrophage": ["C1qa", "C1qb", "C1qc", "Mertk", "Fcgr1", "Cd68", "Aif1", "Ctss", "Tyrobp",
                   "Apoe", "Lgals3", "Marco", "Pparg"],
    "fibroblast": ["Pdgfra", "Col1a1", "Col1a2", "Col3a1", "Dcn", "Lum",
                   "Col5a1", "Col6a1", "Col6a2", "Col14a1", "Dpt", "Pi16", "C7"],
    "neutrophil": ["Ly6g", "Csf3r", "Retnlg", "Ngp", "Camp", "Ltf", "Mmp8", "Mmp9",
                   "S100a8", "S100a9", "Mpo", "Elane", "Cebpe", "Ms4a3"],
    "eosinophil": ["Epx", "Prg2", "Prg3", "Siglecf", "Ccr3", "Alox15",
                   "Ear1", "Ear2", "Ear6", "Il5ra", "Gata1", "Itgam", "Ptprc", "Itga4"],
    "basophil": ["Mcpt8", "Cd200r3", "Clec12a", "Fcer1a", "Ms4a2", "Hdc",
                 "Prss34", "Alox5ap", "Il4", "Ccl3", "Ccl4", "Gata2"],
}
SUBTYPE_MARKERS = {
    "classical_monocyte": ["Ly6c2", "Ccr2", "Sell", "Vcan"],
    "nonclassical_monocyte": ["Fcgr3", "Cx3cr1", "Nr4a1", "Treml4"],
    "cDC1": ["Xcr1", "Clec9a", "Batf3", "Irf8"],
    "cDC2": ["Clec10a", "Sirpa", "Irf4"],
    "cytotoxic_NK": ["Ncr1", "Klrd1", "Nkg7", "Prf1", "Gzmb"],
    "plasma_cell": ["Jchain", "Mzb1", "Xbp1", "Sdc1"],
}
PLASMA_NEW = ["Prdm1", "Irf4", "Derl3", "Tnfrsf17", "Slamf7"]
PLASMA_SPECIFIC = ["Derl3", "Tnfrsf17", "Slamf7", "Sdc1"]
PLASMA_EXTRA = ["Prdm1", "Irf4", "Xbp1", "Cd79a"]


In [4]:
genes = list(dict.fromkeys(
    [g for panel in PANELS.values() for g in panel]
    + [g for panel in SUBTYPE_MARKERS.values() for g in panel]
    + PLASMA_NEW + PLASMA_SPECIFIC + PLASMA_EXTRA
    + ["Sdc1", "Xbp1", "Cd79a", "Cd79b", "Ms4a1"]
))
present = [g for g in genes if g in adata.var_names]

slim = adata[:, present].to_memory()
slim.obs["leiden"] = slim.obs["leiden"].astype(str)
print(f"{len(present)}/{len(genes)} marker genes present; {slim.n_obs:,} cells")


133/137 marker genes present; 133,745 cells


In [5]:
X = np.asarray(slim.X.toarray() if hasattr(slim.X, "toarray") else slim.X, dtype=np.float32)

detected = pd.DataFrame(X > 0, columns=present, index=slim.obs_names)
detected["leiden"] = slim.obs["leiden"].to_numpy()
pct = detected.groupby("leiden", observed=True)[present].mean()


In [6]:
sup = pd.DataFrame({k: (pct[[g for g in v if g in present]] > 0.30).sum(axis=1)
                    for k, v in PANELS.items()})
med = pd.DataFrame({k: pct[[g for g in v if g in present]].median(axis=1)
                    for k, v in PANELS.items()})

In [7]:
rank = np.argsort(-sup.to_numpy(), axis=1)
top1 = sup.columns[rank[:, 0]]
top2 = sup.columns[rank[:, 1]]
s1 = sup.to_numpy()[np.arange(len(sup)), rank[:, 0]]
s2 = sup.to_numpy()[np.arange(len(sup)), rank[:, 1]]

ev = pd.DataFrame({
    "n_cells": slim.obs["leiden"].value_counts().reindex(sup.index).astype(int),
    "top1": top1, "s1": s1, "top2": top2, "s2": s2, "margin": s1 - s2,
    "med1": [round(float(med.loc[c, t]), 2) for c, t in zip(sup.index, top1)],
}, index=sup.index)
ev["ng_median"] = slim.obs.groupby("leiden", observed=True)["n_genes"].median().astype(int)
ev["auto"] = np.where((ev["s1"] >= 5) & (ev["margin"] >= 2), ev["top1"], "REVIEW")
ev

,n_cells,top1,s1,top2,s2,margin,med1,ng_median,auto
leiden,,,,,,,,,
0,4844,endothelial,11,macrophage,2,9,0.89,1940,endothelial
1,658,endothelial,11,dendritic,4,7,0.82,2683,endothelial
10,5881,neutrophil,11,monocyte,9,2,0.56,2646,neutrophil
11,4369,t_cell,6,nk_cell,5,1,0.87,1483,REVIEW
12,2086,nk_cell,10,monocyte,5,5,0.94,1419,nk_cell
13,2013,dendritic,7,macrophage,6,1,0.47,2155,REVIEW
14,2349,macrophage,10,monocyte,5,5,0.93,2246,macrophage
15,527,fibroblast,1,macrophage,1,0,0.07,434,REVIEW
16,6448,monocyte,9,macrophage,6,3,0.75,1196,monocyte


In [8]:
MANUAL = {
    "0": ("endothelial", "Pecam1/Cdh5/Kdr/Flt1/Emcn/Cldn5/Esam/Egfl7 等 11 个阳性"),
    "1": ("endothelial", "同上 11 个 + Vwf"),
    "19": ("endothelial", "10 个阳性，Kidney"),
    "39": ("endothelial", "11 个阳性，Liver"),
    "41": ("endothelial", "12/12 全阳性，Lung"),
    "4": ("fibroblast", "fibroblast panel 12/13（Pdgfra/Col1a1/Col1a2/Col3a1/Dcn/Lum/Dpt/Pi16）；脂肪组织"),
    "6": ("fibroblast", "Col1a2/Col3a1/Dcn/Col6a1 4 个（跨组织间质）；support<5 故人工确认"),
    "17": ("fibroblast", "9 个阳性，Lung"),
    "20": ("fibroblast", "12/13 阳性，Heart 成纤维细胞"),
    "2": ("b_cell", "Cd79a/Cd79b/Cd19/Ms4a1/Cd22/Bank1/Cd37/Spib 8 个"),
    "3": ("b_cell", "同上 8 个"),
    "18": ("b_cell", "同上 8 个"),
    "5": ("t_cell", "Cd3d/e/g+Lck/Lat/Cd247 6 个"),
    "11": ("t_cell", "6 个 CD3 复合体基因 + 细胞毒程序(Klrd1/Nkg7/Prf1)，判 T 不判 NK"),
    "35": ("t_cell", "同 c11"),
    "12": ("nk_cell", "Ncr1/Klrd1/Klrb1c/Klrk1/Eomes/Il2rb/Nkg7/Prf1 等 10/10"),
    "13": ("dendritic", "Flt3/Itgax/Clec9a/Sirpa+MHC-II 7 个；DC/巨噬细胞交界，取 DC"),
    "14": ("macrophage", "Fcgr1/C1qa/b/c/Cd68/Aif1/Ctss/Tyrobp 等 10 个，Kidney"),
    "30": ("macrophage", "10 个，med1=1.00，Liver(Kupffer 候选)"),
    "31": ("macrophage", "9 个，Lung"),
    "38": ("macrophage", "10 个，Liver"),
    "40": ("macrophage", "Marco 0.44/Pparg 0.37/Mrc1 0.96/Itgax 0.72，C1q 阴性 → 肺泡巨噬；补基因后 macrophage 7 : monocyte 5"),
    "9": ("monocyte", "Ly6c2 0.69/Ccr2 0.69/Csf1r 0.91/Lyz2 1.00，无 Ngp/Camp，Lung"),
    "16": ("monocyte", "Ly6c2 0.85/Ccr2 0.73/Csf1r 0.57/Plac8 0.99，无 Ngp/Camp，Lung"),
    "32": ("neutrophil", "Ngp/Camp 1.00、Retnlg 0.97、Ltf 0.94、Mmp8 0.77、Mmp9 0.90（成熟中性粒）"),
    "33": ("neutrophil", "Ngp/Camp 1.00 + Mpo 0.62/Elane 0.78/Cebpe 0.99/Ms4a3 0.84（未成熟/粒系祖）"),
    "10": ("neutrophil", "Ngp 0.94/Camp 0.95/Mpo 0.51/Elane 0.68（未成熟中性粒）"),
    "21": ("pDC_enriched_unresolved", "pDC 核心富集；不能证明全部为纯 pDC"),
    "24": ("erythroid", "Alas2/Gypa/Slc4a1 接近 100%，Csf3r 约 3%；免疫主线排除"),
    "25": ("b_cell_precursor", "Cd79a/b 与 Vpreb/Igll1/Dntt 联合证据；保留前体区别"),
    "26": ("erythroid_neutrophil_mixed", "红系与粒系核心同时表达；机制未定，排除主线"),
    "27": ("cycling_erythroid", "红系程序加完整增殖程序"),
    "22": ("basophil", "Mcpt8 0.79/Cd200r3 0.92/Fcer1a 0.93/Hdc 1.00/Alox5ap 0.99/Gata2 0.97，Lung n=120"),
    "8": ("plasma_cell", "Derl3 0.96/Sdc1 0.94/Slamf7 0.83/Tnfrsf17 0.65，组合证据 97.9%"),
    "23": ("ambiguous", "851/852 细胞至少两个 Csf3r/Retnlg/Mmp8/Mmp9 阳性，支持成熟中性粒；仅作恢复对照，不进主线"),
    "7": ("unassigned", "macrophage 2 / endothelial 2，med 0.04（Kidney 上皮）"),
    "15": ("unassigned", "endothelial 1 / macrophage 1；n_genes 434 低质量"),
    "28": ("unassigned", "dendritic 2（仅 MHC-II）/ macrophage 1，med 0.00（Kidney 上皮）"),
    "29": ("unassigned", "macrophage 2（Apoe/Lgals3）；Tnnt2 1.00 为心肌细胞"),
    "34": ("unassigned", "macrophage 2 / dendritic 2，med 0.07"),
    "36": ("unassigned", "macrophage 1（仅 Apoe），Liver（肝细胞）"),
    "37": ("unassigned", "macrophage 1（仅 Apoe），Liver（肝细胞）"),
}

In [9]:
missing = [c for c in ev.index if c not in MANUAL]
extra = [c for c in MANUAL if c not in ev.index]
assert not missing and not extra, (missing, extra)

slim.obs["cell_type"] = slim.obs["leiden"].map({c: v[0] for c, v in MANUAL.items()})
slim.obs["auto_label"] = slim.obs["leiden"].map(ev["auto"])
assert slim.obs["cell_type"].notna().all()

slim.obs["cell_type"].value_counts()

cell_type
b_cell                        26609
neutrophil                    24872
unassigned                    14788
t_cell                        12041
monocyte                      11314
endothelial                   10421
fibroblast                     7867
macrophage                     6813
cycling_erythroid              4992
erythroid                      3506
pDC_enriched_unresolved        2295
nk_cell                        2086
dendritic                      2013
plasma_cell                    1441
b_cell_precursor               1399
ambiguous                       852
erythroid_neutrophil_mixed      316
basophil                        120
Name: count, dtype: int64

In [10]:
ev["manual"] = [MANUAL[c][0] for c in ev.index]
ev["rationale"] = [MANUAL[c][1] for c in ev.index]
ev["tissue_top"] = [
    "/".join(f"{k} {v:.0%}" for k, v in
             slim.obs.loc[slim.obs["leiden"] == c, "tissue"].value_counts(normalize=True).head(2).items())
    for c in ev.index
]
ev["n_mice"] = [slim.obs.loc[slim.obs["leiden"] == c, "mouse.id"].nunique() for c in ev.index]

def confidence(c):
    label = MANUAL[c][0]
    if label in ("ambiguous", "unassigned"):
        return "na"
    if label == "plasma_cell":
        return "high"
    if c == "15":
        return "low"
    if ev.loc[c, "s1"] >= 8 and ev.loc[c, "margin"] >= 3:
        return "high"
    return "medium"

ev["confidence"] = [confidence(c) for c in ev.index]
slim.obs["annotation_confidence"] = slim.obs["leiden"].map(ev["confidence"])

audit = ev[["n_cells", "n_mice", "tissue_top", "ng_median", "top1", "s1", "top2", "s2",
            "margin", "med1", "auto", "manual", "confidence", "rationale"]]
audit.index.name = "cluster"
audit.to_csv(RES / "v2/cluster_annotation.tsv", sep="\t")
audit

,n_cells,n_mice,tissue_top,ng_median,top1,s1,top2,s2,margin,med1,auto,manual,confidence,rationale
cluster,,,,,,,,,,,,,,
0,4844,15,Heart_and_Aorta 73%/Fat 26%,1940,endothelial,11,macrophage,2,9,0.89,endothelial,endothelial,high,Pecam1/Cdh5/Kdr/Flt1/Emcn/Cldn5/Esam/Egfl7 等 1...
1,658,18,Heart_and_Aorta 87%/Lung 6%,2683,endothelial,11,dendritic,4,7,0.82,endothelial,endothelial,high,同上 11 个 + Vwf
10,5881,19,spleen/marrow 97%/Kidney 3%,2646,neutrophil,11,monocyte,9,2,0.56,neutrophil,neutrophil,medium,Ngp 0.94/Camp 0.95/Mpo 0.51/Elane 0.68（未成熟中性粒）
11,4369,21,spleen/marrow 49%/Lung 29%,1483,t_cell,6,nk_cell,5,1,0.87,REVIEW,t_cell,medium,6 个 CD3 复合体基因 + 细胞毒程序(Klrd1/Nkg7/Prf1)，判 T 不判 NK
12,2086,22,Lung 57%/spleen/marrow 32%,1419,nk_cell,10,monocyte,5,5,0.94,nk_cell,nk_cell,high,Ncr1/Klrd1/Klrb1c/Klrk1/Eomes/Il2rb/Nkg7/Prf1 ...
13,2013,20,spleen/marrow 44%/Kidney 20%,2155,dendritic,7,macrophage,6,1,0.47,REVIEW,dendritic,medium,Flt3/Itgax/Clec9a/Sirpa+MHC-II 7 个；DC/巨噬细胞交界，取 DC
14,2349,19,Kidney 34%/Fat 33%,2246,macrophage,10,monocyte,5,5,0.93,macrophage,macrophage,high,Fcgr1/C1qa/b/c/Cd68/Aif1/Ctss/Tyrobp 等 10 个，Ki...
15,527,14,Heart_and_Aorta 69%/Fat 12%,434,fibroblast,1,macrophage,1,0,0.07,REVIEW,unassigned,na,endothelial 1 / macrophage 1；n_genes 434 低质量
16,6448,16,Lung 99%/spleen/marrow 1%,1196,monocyte,9,macrophage,6,3,0.75,monocyte,monocyte,high,Ly6c2 0.85/Ccr2 0.73/Csf1r 0.57/Plac8 0.99，无 N...


In [11]:
GROUPS = {
    "lymphoid": {"t_cell": PANELS["t_cell"], "b_cell": PANELS["b_cell"],
                 "nk_cell": PANELS["nk_cell"], "plasma": PLASMA_SPECIFIC + ["Prdm1", "Irf4", "Xbp1"]},
    "myeloid": {"monocyte": PANELS["monocyte"], "dendritic": PANELS["dendritic"],
                "macrophage": [g for g in PANELS["macrophage"] if g in present]},
    "structural": {"endothelial": PANELS["endothelial"], "fibroblast": PANELS["fibroblast"]},
    "granulocyte": {"neutrophil": PANELS["neutrophil"], "eosinophil": PANELS["eosinophil"],
                    "basophil": PANELS["basophil"]},
}

for name, panels in GROUPS.items():
    panels = {k: [g for g in v if g in present] for k, v in panels.items()}
    sc.pl.dotplot(slim, panels, groupby="leiden", standard_scale="var", cmap="Reds",
                  figsize=(13, 11), show=False)
    plt.savefig(FIG / f"05_dotplot_{name}.png", dpi=160, bbox_inches="tight")
    plt.close()

In [12]:
slim.obs["cell_type"] = pd.Categorical(
    slim.obs["cell_type"],
    categories=["endothelial", "fibroblast", "macrophage", "monocyte", "neutrophil", "basophil",
                "dendritic", "t_cell", "nk_cell", "b_cell", "plasma_cell", "b_cell_precursor",
                "pDC_enriched_unresolved", "erythroid", "cycling_erythroid",
                "erythroid_neutrophil_mixed", "ambiguous", "unassigned"])

sc.pl.umap(slim, color="cell_type", frameon=False, palette="tab20", size=2, alpha=0.8,
           title="cell_type", show=False)
plt.savefig(FIG / "05_umap_cell_type.png", dpi=200, bbox_inches="tight")
plt.close()

sc.pl.umap(slim, color="annotation_confidence", frameon=False, size=2, alpha=0.8,
           title="annotation confidence", show=False)
plt.savefig(FIG / "05_umap_confidence.png", dpi=200, bbox_inches="tight")
plt.close()

sc.pl.umap(slim, color="tissue", frameon=False, size=2, alpha=0.8, title="tissue", show=False)
plt.savefig(FIG / "05_umap_tissue.png", dpi=200, bbox_inches="tight")
plt.close()

In [13]:
PLOT = {
    "endothelial": PANELS["endothelial"],
    "fibroblast": PANELS["fibroblast"],
    "macrophage": [g for g in PANELS["macrophage"] if g in present],
    "monocyte": PANELS["monocyte"],
    "dendritic": PANELS["dendritic"],
    "t_cell": PANELS["t_cell"],
    "nk_cell": PANELS["nk_cell"],
    "b_cell": PANELS["b_cell"],
    "plasma_cell": PLASMA_SPECIFIC + ["Prdm1", "Irf4", "Xbp1"],
    "neutrophil": PANELS["neutrophil"],
    "eosinophil": PANELS["eosinophil"],
    "basophil": PANELS["basophil"],
}

for cell_type, genes in PLOT.items():
    genes = [g for g in genes if g in present]
    sc.pl.umap(slim, color=genes, ncols=4, vmax="p99", cmap="Reds",
               frameon=False, size=3, show=False)
    plt.suptitle(f"{cell_type} markers", y=1.01, fontsize=13)
    plt.savefig(FIG / f"05_umap_markers_{cell_type}.png", dpi=150, bbox_inches="tight")
    plt.close()

In [14]:
slim = slim[~slim.obs["cell_type"].isin(["unassigned", "ambiguous"])].copy()

CT_SCOPE = {"endothelial": "endothelial", "macrophage": "macrophage"}
slim.obs["ct_scope"] = pd.Categorical(
    slim.obs["cell_type"].map(CT_SCOPE),
    categories=["endothelial", "macrophage"], ordered=True)
assert slim.obs["ct_scope"].notna().any()

print(f"{slim.n_obs:,} cells retained")

118,105 cells retained


In [15]:
lbl = slim.obs[["leiden", "tissue", "mouse.id", "cell_type", "ct_scope",
                "annotation_confidence", "auto_label"]].copy()
lbl.index.name = "cell"
lbl.to_csv(RES / "cell_labels.tsv.gz", sep="\t", compression="gzip")

summary = (lbl.groupby("cell_type", observed=True)
           .agg(n_cells=("leiden", "size"), n_mice=("mouse.id", "nunique"))
           .reset_index().sort_values("n_cells", ascending=False))
summary.to_csv(RES / "cell_type_summary.tsv", sep="\t", index=False)

scope_summary = (lbl.dropna(subset=["ct_scope"])
                 .groupby(["ct_scope", "cell_type"], observed=True)
                 .agg(n_cells=("leiden", "size"), n_mice=("mouse.id", "nunique"))
                 .reset_index().sort_values("n_cells", ascending=False))
scope_summary.to_csv(RES / "cell_type_summary_scope.tsv", sep="\t", index=False)

summary

,cell_type,n_cells,n_mice
9,b_cell,26609,22
4,neutrophil,24872,20
7,t_cell,12041,21
3,monocyte,11314,22
0,endothelial,10421,20
1,fibroblast,7867,21
2,macrophage,6813,20
14,cycling_erythroid,4992,19
13,erythroid,3506,19
12,pDC_enriched_unresolved,2295,21


In [16]:
full = sc.read_h5ad("../data_interim/dimensionality_reduced.h5ad")
full = full[slim.obs_names].copy()

for col in ["cell_type", "ct_scope", "annotation_confidence", "auto_label"]:
    full.obs[col] = slim.obs[col]

full.write_h5ad("../data_interim/cell_type_annotated.h5ad")

In [17]:
X = np.asarray(slim.X.toarray() if hasattr(slim.X, "toarray") else slim.X, dtype=np.float32)

marker = pd.DataFrame(X, columns=present, index=slim.obs_names)
for col in ["mouse.id", "tissue", "cell_type"]:
    marker[col] = slim.obs[col].astype(str).to_numpy()

sample_mean = marker.groupby(["mouse.id", "tissue", "cell_type"], observed=True)[present].mean()
expression = sample_mean.groupby("cell_type", observed=True).mean()
detection = (marker[present].gt(0)
             .groupby(marker["cell_type"], observed=True).mean()
             .reindex(expression.index))

expression.to_csv("../figure_data/celltype_marker_mean_log_expression.tsv", sep="\t")
detection.to_csv("../figure_data/celltype_marker_fraction_detected.tsv", sep="\t")

panel_audit = pd.DataFrame([
    dict(panel=panel, gene=gene, present=gene in present, source="05 PANELS / PLASMA_SPECIFIC")
    for panel, genes in {**PANELS, "plasma_cell": PLASMA_SPECIFIC + PLASMA_NEW}.items()
    for gene in genes
])
panel_audit.to_csv("../results/qc/marker_panel_audit.tsv", sep="\t", index=False)

scaled = expression.divide(expression.max(axis=0).replace(0, np.nan), axis=1).fillna(0)
fig, ax = plt.subplots(figsize=(16, 7), layout="constrained")
for i in range(len(expression)):
    ax.scatter(np.arange(len(present)), np.repeat(i, len(present)),
               s=8 + 170 * detection.iloc[i].to_numpy(),
               c=scaled.iloc[i].to_numpy(), vmin=0, vmax=1, cmap="Reds",
               edgecolors="#999999", linewidths=.25)
ax.set_xticks(range(len(present)), present, rotation=90, fontsize=8)
ax.set_yticks(range(len(expression)), [str(a) for a in expression.index], fontsize=8)
ax.invert_yaxis()
ax.set_title("Current annotations | marker panels\n"
             "Colour: mean log expression / gene maximum; size: detected fraction; "
             "equal mouse-ID x tissue weight", fontsize=11)
fig.colorbar(plt.cm.ScalarMappable(norm=plt.Normalize(0, 1), cmap="Reds"),
             ax=ax, label="Within-gene scaled mean", shrink=.6)
fig.savefig("../figures/qc/celltype_marker_check.pdf")
fig.savefig("../figures/qc/celltype_marker_check.png", dpi=140)
plt.close(fig)


In [18]:
AGE_GROUP = {1: "young", 3: "young", 18: "middle", 21: "middle", 24: "middle", 30: "old"}

cohort = slim.obs.copy()
cohort["age_months"] = pd.to_numeric(cohort["age"].astype(str).str.replace("m", "", regex=False))
cohort["age_group"] = cohort["age_months"].map(AGE_GROUP)
assert cohort["age_group"].notna().all()

scoped = cohort[cohort["tissue"].isin(["Heart_and_Aorta", "Kidney", "Liver", "Lung"])]
cohort_matrix = (scoped.groupby(["mouse.id", "tissue", "ct_scope", "age_months", "age_group", "sex"],
                                observed=True)
                 .size().rename("n_cells").reset_index()
                 .rename(columns={"mouse.id": "mouse_id"}))

assert cohort_matrix[["mouse_id", "tissue", "ct_scope"]].drop_duplicates().shape[0] == len(cohort_matrix)
assert cohort_matrix["tissue"].nunique() == 4
assert cohort_matrix["ct_scope"].nunique() == 2

cohort_matrix["sampling_unit"] = "mouse.id identifier; composite identifiers retained and audited"
cohort_matrix.to_csv("../results/qc/final_cohort_matrix.csv", index=False)

cohort_matrix.groupby(["tissue", "ct_scope"], observed=True)["n_cells"].sum().unstack()


ct_scope,endothelial,macrophage
tissue,,
Heart_and_Aorta,4120,720
Kidney,1526,822
Liver,673,2411
Lung,2768,2089


In [19]:
print(f"cells: {slim.n_obs:,} | clusters: {ev.shape[0]} | labels: {summary.shape[0]}")
print(f"cohort units: {len(cohort_matrix):,} | cohort cells: {int(cohort_matrix.n_cells.sum()):,}")

cells: 118,105 | clusters: 42 | labels: 16
cohort units: 101 | cohort cells: 15,129
